# 03 — From EUDR Risk Score to Supply Volatility

**The Bridge Notebook** — connecting the sustainability and data science teams.

This is the unique insight of this project: EUDR deforestation risk is not just
a compliance problem — it is a **supply chain risk signal**.

A supplier with high deforestation risk faces:
- Potential EU market ban (direct supply disruption)
- Reputational risk leading to contract termination
- Regulatory scrutiny causing delays
- Pressure to change farming practices (yield uncertainty)

Therefore: **high EUDR risk → higher probability of supply disruption → should be
incorporated into demand/inventory planning as a supply volatility feature.**

This is what most FMCG companies do not yet do.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

## 1. Load Supplier Risk Scores (from notebook 01)

In [ ]:
# Simulated supplier portfolio with EUDR risk scores
# In production: load from 01_gee_forest_data_pipeline output
suppliers = pd.DataFrame([
    {'supplier_id': 'COL-001', 'commodity': 'coffee',
     'annual_volume_tons': 45,  'eudr_risk_score': 0.12},
    {'supplier_id': 'COL-002', 'commodity': 'coffee',
     'annual_volume_tons': 120, 'eudr_risk_score': 0.18},
    {'supplier_id': 'COL-003', 'commodity': 'coffee',
     'annual_volume_tons': 67,  'eudr_risk_score': 0.22},
    {'supplier_id': 'COL-004', 'commodity': 'coffee',
     'annual_volume_tons': 89,  'eudr_risk_score': 0.15},
    {'supplier_id': 'ETH-001', 'commodity': 'coffee',
     'annual_volume_tons': 95,  'eudr_risk_score': 0.09},
    {'supplier_id': 'ETH-002', 'commodity': 'coffee',
     'annual_volume_tons': 210, 'eudr_risk_score': 0.11},
    {'supplier_id': 'ETH-003', 'commodity': 'coffee',
     'annual_volume_tons': 155, 'eudr_risk_score': 0.14},
    {'supplier_id': 'BRA-001', 'commodity': 'sugar',
     'annual_volume_tons': 500, 'eudr_risk_score': 0.48},
])

# Volume-weighted portfolio risk score per commodity
def portfolio_risk(df, commodity):
    sub = df[df['commodity'] == commodity]
    return np.average(sub['eudr_risk_score'], weights=sub['annual_volume_tons'])

coffee_risk = portfolio_risk(suppliers, 'coffee')
sugar_risk  = portfolio_risk(suppliers, 'sugar')

print(f'Volume-weighted EUDR risk — Coffee: {coffee_risk:.3f} | Sugar: {sugar_risk:.3f}')

## 2. Model: EUDR Risk → Supply Disruption Probability

In [ ]:
def eudr_risk_to_supply_disruption_prob(risk_score):
    """
    Convert EUDR deforestation risk score to estimated annual
    supply disruption probability.

    Mapping rationale:
    - Low risk (0-0.2):    2-8% disruption probability
    - Medium risk (0.2-0.4): 8-25% disruption probability
    - High risk (0.4-1.0): 25-70% disruption probability

    This is a simplified linear model. In production, calibrate
    against historical supplier disruption events.
    """
    # Sigmoid-like mapping from risk score to disruption probability
    disruption_prob = 0.02 + 0.68 * (1 / (1 + np.exp(-8 * (risk_score - 0.35))))
    return np.clip(disruption_prob, 0.02, 0.70)


# Show the mapping
risk_range = np.linspace(0, 1, 100)
disruption_range = eudr_risk_to_supply_disruption_prob(risk_range)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(risk_range, disruption_range * 100, color='#185FA5', linewidth=2)
ax.axvline(x=coffee_risk, color='#1D9E75', linestyle='--',
           label=f'Coffee portfolio risk: {coffee_risk:.2f}')
ax.axvline(x=sugar_risk, color='#D85A30', linestyle='--',
           label=f'Sugar portfolio risk: {sugar_risk:.2f}')
ax.set_xlabel('EUDR Deforestation Risk Score')
ax.set_ylabel('Annual Supply Disruption Probability (%)')
ax.set_title('EUDR Risk → Supply Disruption Probability Mapping', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Coffee disruption probability: {eudr_risk_to_supply_disruption_prob(coffee_risk)*100:.1f}%')
print(f'Sugar disruption probability:  {eudr_risk_to_supply_disruption_prob(sugar_risk)*100:.1f}%')

## 3. Incorporate Supply Risk into Inventory Safety Stock

In [ ]:
def calculate_safety_stock(mean_demand, std_demand, lead_time_weeks,
                            service_level_z, supply_disruption_prob,
                            disruption_duration_weeks=4):
    """
    Safety stock incorporating EUDR-linked supply disruption risk.

    Standard safety stock + disruption buffer:
    SS = z * σ * sqrt(LT) + P(disruption) * mean_demand * disruption_duration

    Parameters
    ----------
    mean_demand : float — weekly units
    std_demand : float — weekly demand std dev
    lead_time_weeks : int — supplier lead time
    service_level_z : float — z-score for service level (1.64 = 95%)
    supply_disruption_prob : float — annual disruption probability
    disruption_duration_weeks : int — expected disruption length
    """
    # Standard safety stock for demand variability
    ss_demand = service_level_z * std_demand * np.sqrt(lead_time_weeks)

    # Additional buffer for supply disruption risk
    # Weekly disruption prob = 1 - (1 - annual_prob)^(1/52)
    weekly_disruption_prob = 1 - (1 - supply_disruption_prob) ** (1/52)
    ss_disruption = weekly_disruption_prob * mean_demand * disruption_duration_weeks

    total_ss = ss_demand + ss_disruption

    return {
        'ss_demand_variability': round(ss_demand, 0),
        'ss_disruption_buffer': round(ss_disruption, 0),
        'total_safety_stock': round(total_ss, 0),
        'disruption_buffer_pct': round(ss_disruption / total_ss * 100, 1)
    }


# Calculate for coffee ingredient
coffee_disruption_prob = eudr_risk_to_supply_disruption_prob(coffee_risk)
sugar_disruption_prob  = eudr_risk_to_supply_disruption_prob(sugar_risk)

coffee_ss = calculate_safety_stock(
    mean_demand=180, std_demand=25, lead_time_weeks=8,
    service_level_z=1.64,  # 95% service level
    supply_disruption_prob=coffee_disruption_prob
)

sugar_ss = calculate_safety_stock(
    mean_demand=500, std_demand=60, lead_time_weeks=4,
    service_level_z=1.64,
    supply_disruption_prob=sugar_disruption_prob
)

print('=== Coffee Ingredient Safety Stock ===')
for k, v in coffee_ss.items(): print(f'  {k}: {v}')
print()
print('=== Sugar Ingredient Safety Stock ===')
for k, v in sugar_ss.items(): print(f'  {k}: {v}')

## 4. The Business Case — What This Means for Red Bull

In [ ]:
# Scenario: what happens to safety stock if we improve supplier EUDR compliance?

scenarios = {
    'Current (BRA-001 high risk)': sugar_disruption_prob,
    'Switch to low-risk supplier': eudr_risk_to_supply_disruption_prob(0.10),
    'Supplier gets certified': eudr_risk_to_supply_disruption_prob(0.05),
}

results = []
for scenario, prob in scenarios.items():
    ss = calculate_safety_stock(500, 60, 4, 1.64, prob)
    results.append({
        'scenario': scenario,
        'disruption_prob_pct': round(prob * 100, 1),
        'total_safety_stock_tons': ss['total_safety_stock'],
        'disruption_buffer_tons': ss['ss_disruption_buffer']
    })

results_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(results_df['scenario'], results_df['total_safety_stock_tons'],
               color=['#D85A30', '#EF9F27', '#1D9E75'], edgecolor='white', height=0.5)
ax.set_xlabel('Required Safety Stock (tons)')
ax.set_title('EUDR Compliance Improvement → Safety Stock Reduction', fontsize=12)
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, results_df['total_safety_stock_tons']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f} tons', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print(results_df.to_string(index=False))

## Key Takeaways

This notebook demonstrates the **full value chain** of EUDR compliance work:

```
Satellite data → Deforestation risk score → Supply disruption probability
    → Inventory safety stock adjustment → Working capital impact
```

**The business case for Red Bull:**
- Improving supplier EUDR compliance reduces required safety stock
- Reduced safety stock = lower working capital tied up in inventory
- This can be quantified in euros — making sustainability financially tangible

**This is the conversation bridge between:**
- The Sustainability Procurement team (who own EUDR compliance)
- The Data Science team (who own supply chain optimization)
- Finance (who own working capital targets)

**Next:** Integrated tool + dashboard (`04_integrated_tool/`)
